In [11]:
from pyspark.sql.functions import explode
import json

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 13, Finished, Available, Finished)

In [12]:
#file_date='2025-07-11'
df = spark.read.option("multiline", "true").json(f"Files/Bing-lastet-news/{file_date}.json")
# df now is a Spark DataFrame containing JSON data from "Files/Bing-lastet-news/2025-07-08.json".
display(df)

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, af31f069-18db-4e68-9b25-560071a74fdb)

In [13]:
df=df.select("Value")
display(df)

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, fbbdcfa6-edf7-46e6-9195-2e937e982724)

In [14]:
df_exploded=df.select(explode(df["Value"]).alias("json_object"))

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 16, Finished, Available, Finished)

In [15]:
json_list=df_exploded.toJSON().collect()

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 17, Finished, Available, Finished)

In [16]:
title=[]
description=[]
category=[]
url=[]
image=[]
provider=[]
datePublished=[]

for i in json_list:
    try:
        article=json.loads(i)
        
        if article["json_object"].get("category") and article["json_object"].get("image",{}).get("thumbnail",{}).get("contentUrl"):
            title.append(article["json_object"]["name"])
            description.append(article["json_object"]["description"])
            category.append(article["json_object"]["category"])
            url.append(article["json_object"]["url"])
            image.append(article["json_object"]["image"]["thumbnail"]["contentUrl"])
            provider.append(article["json_object"]["provider"][0]["name"])
            datePublished.append(article["json_object"]["datePublished"])
    except Exception as e:
        print(f"Error Processing JSON object :{e}")


StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 18, Finished, Available, Finished)

In [17]:
from pyspark.sql.types import StructType,StructField,StringType,DateType

# combine the lists
data=list(zip(title,description,category,url,image,provider,datePublished))

# create a schema

schema1=StructType([
    StructField("title",StringType(),True),
    StructField("description",StringType(),True),
    StructField("category",StringType(),True),
    StructField("url",StringType(),True),
    StructField("image",StringType(),True),
    StructField("provider",StringType(),True),
    StructField("datePublished",StringType(),True)
])

# Create a Schema

df_cleaned =spark.createDataFrame(data,schema=schema1)

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 19, Finished, Available, Finished)

In [18]:
from pyspark.sql.functions import date_format, to_date

df_cleaned_final=df_cleaned.withColumn("datePublished",date_format(to_date("datePublished"),"yyyy-MM-dd"))

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 20, Finished, Available, Finished)

In [19]:
display(df_cleaned_final)

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 21, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a50192c6-a397-4b70-b530-ae944e68c416)

In [20]:
from pyspark.sql.utils import AnalysisException

try:
    tableName="bing_lake_db.tbl_latest_news"
    df_cleaned_final.write.format("delta").saveAsTable(tableName)

except AnalysisException:
    print("table already Exist")
    df_cleaned_final.createOrReplaceTempView("vw_tbl_latest_news")
    spark.sql(f""" MERGE INTO {tableName} as Target_table
                   USING vw_tbl_latest_news AS SourceTable
                   on SourceTable.url= Target_table.url

                   WHEN MATCHED AND
                   SourceTable.title<> Target_table.title OR
                   SourceTable.description<> Target_table.description OR
                   SourceTable.category<> Target_table.category OR
                   SourceTable.image<> Target_table.image OR
                   SourceTable.provider<> Target_table.provider OR
                   SourceTable.datePublished<> Target_table.datePublished

                   THEN UPDATE SET *

                   WHEN NOT MATCHED THEN INSERT *
              """)

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 22, Finished, Available, Finished)

table already Exist


In [21]:
%%sql

select count(*) from bing_lake_db.tbl_latest_news

StatementMeta(, f0e4fc93-a35e-4c85-80a0-92798a46929b, 23, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>